# Legacy exploratory notebook — superseded
This notebook predates the audited engine and is retained only as development history. Its outputs are not current research results. In particular, early backtest notebooks contain obsolete timing, missing-return filtering, and drawdown calculations. Do not use them to validate performance or overwrite the bundled data. Use the root Python entrypoints and [methodology](../docs/METHODOLOGY.md).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [2]:
prices = pd.read_parquet("../data/raw/us_stock.parquet")
benchmark = pd.read_parquet("../data/raw/benchmark.parquet")

prices.head()

Price           Close                                                        \
Ticker           AAPL       ABBV     AMZN        COST        CVX      GOOGL   
Date                                                                          
2015-01-02  24.171757  41.119228  15.4260  113.634468  68.898331  26.244781   
2015-01-05  23.490793  40.345398  15.1095  112.342575  66.144371  25.744715   
2015-01-06  23.493008  40.145714  14.7645  113.827087  66.113747  25.109352   
2015-01-07  23.822432  41.768253  14.9210  115.809135  66.058678  25.035503   
2015-01-08  24.737745  42.205105  15.0230  116.804153  67.570320  25.122732   

Price                                                   ...    Volume  \
Ticker             HD        JPM         KO         MA  ...      META   
Date                                                    ...             
2015-01-02  78.780350  46.066788  29.390272  79.714668  ...  18177500   
2015-01-05  77.127525  44.636658  29.390272  77.472488  ...  26452200   
2015-01-06  76.891396  43.479259  29.613443  77.305023  ...  27399300   
2015-01-07  79.526817  43.545616  29.983091  78.507530  ...  22045300   
2015-01-08  81.286285  44.518688  30.345758  79.728683  ...  23961000   

Price                                                                 \
Ticker          MSFT       NVDA      PEP       PG      TSLA      UNH   
Date                                                                   
2015-01-02  27913900  113680000  3545700  7251400  71466000  3060900   
2015-01-05  39673900  197952000  6441000  8626100  80527500  4679000   
2015-01-06  36447900  197764000  6195000  7791200  93928500  3468300   
2015-01-07  29114100  321808000  6526300  5986600  44526000  3225800   
2015-01-08  29645200  283780000  7131600  6823300  51637500  5346100   

Price                                     
Ticker             V       WMT       XOM  
Date                                      
2015-01-02   8389600  13505400  10220400  
2015-01-05  12751200  20937000  18502400  
2015-01-06  11070000  24615300  16670700  
2015-01-07   9346800  25495200  13590700  
2015-01-08  10443200  38140800  15487500  

[5 rows x 100 columns]

In [3]:
close = prices["Close"].copy()
benchmark_close = benchmark["Close"].copy()

close.head()

Ticker,AAPL,ABBV,AMZN,COST,CVX,GOOGL,HD,JPM,KO,MA,META,MSFT,NVDA,PEP,PG,TSLA,UNH,V,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,
2015-01-02,24.171757,41.119228,15.4260,113.634468,68.898331,26.244781,78.780350,46.066788,29.390272,79.714668,77.767090,39.681736,0.482423,66.582962,65.070015,14.620667,83.352692,60.974831,23.127544,57.145546
2015-01-05,23.490793,40.345398,15.1095,112.342575,66.144371,25.744715,77.127525,44.636658,29.390272,77.472488,76.518059,39.316849,0.474275,66.082344,64.760620,14.006000,81.979774,59.628880,23.060240,55.581936
2015-01-06,23.493008,40.145714,14.7645,113.827087,66.113747,25.109352,76.891396,43.479259,29.613443,77.305023,75.487091,38.739765,0.459896,65.581779,64.465637,14.085333,81.814339,59.244663,23.237934,55.286457
2015-01-07,23.822432,41.768253,14.9210,115.809135,66.058678,25.035503,79.526817,43.545616,29.983091,78.507530,75.487091,39.231953,0.458697,67.499466,64.803802,14.063333,82.649704,60.038418,23.854490,55.846664
2015-01-08,24.737745,42.205105,15.0230,116.804153,67.570320,25.122732,81.286285,44.518688,30.345758,79.728683,77.499428,40.386101,0.475952,68.726234,65.544846,14.041333,86.594841,60.843693,24.357964,56.776207


In [4]:
print(close.shape)
print(close.isna().sum().sort_values(ascending=False).head(10))

(2766, 20)
Ticker
AAPL    0
ABBV    0
WMT     0
V       0
UNH     0
TSLA    0
PG      0
PEP     0
NVDA    0
MSFT    0
dtype: int64


In [5]:
returns = close.pct_change(fill_method=None)

returns.head()

Ticker,AAPL,ABBV,AMZN,COST,CVX,GOOGL,HD,JPM,KO,MA,META,MSFT,NVDA,PEP,PG,TSLA,UNH,V,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,
2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,-0.028172,-0.018819,-0.020517,-0.011369,-0.039971,-0.019054,-0.020980,-0.031045,0.000000,-0.028128,-0.016061,-0.009195,-0.016890,-0.007519,-0.004755,-0.042041,-0.016471,-0.022074,-0.002910,-0.027362
2015-01-06,0.000094,-0.004949,-0.022833,0.013214,-0.000463,-0.024679,-0.003062,-0.025929,0.007593,-0.002162,-0.013474,-0.014678,-0.030318,-0.007575,-0.004555,0.005664,-0.002018,-0.006443,0.007706,-0.005316
2015-01-07,0.014022,0.040416,0.010600,0.017413,-0.000833,-0.002941,0.034275,0.001526,0.012482,0.015555,0.000000,0.012705,-0.002606,0.029241,0.005246,-0.001562,0.010210,0.013398,0.026532,0.010133
2015-01-08,0.038422,0.010459,0.006836,0.008592,0.022883,0.003484,0.022124,0.022346,0.012096,0.015555,0.026658,0.029419,0.037618,0.018174,0.011435,-0.001564,0.047733,0.013413,0.021106,0.016645


In [6]:
annual_return = returns.mean() * 252

annual_return.sort_values(ascending=False).head(10)

annual_volatility = returns.std() * np.sqrt(252)

annual_volatility.sort_values(ascending=False).head(10)

sharpe = annual_return / annual_volatility

sharpe.sort_values(ascending=False).head(10)

wealth = (1 + returns).cumprod()

running_max = wealth.cummax()

drawdown = wealth / running_max - 1

max_drawdown = drawdown.min()

max_drawdown.sort_values().head(10)

Ticker
META    -0.767361
TSLA    -0.736322
NVDA    -0.663351
UNH     -0.613909
XOM     -0.613425
AMZN    -0.561453
CVX     -0.557739
ABBV    -0.450898
GOOGL   -0.443200
JPM     -0.436265
dtype: float64

In [7]:
momentum_20 = close.pct_change(20)
momentum_60 = close.pct_change(60)
momentum_120 = close.pct_change(120)

In [8]:
factor_table = pd.DataFrame({
    "momentum_20d": momentum_20.iloc[-1],
    "momentum_60d": momentum_60.iloc[-1],
    "momentum_120d": momentum_120.iloc[-1],
    "volatility": annual_volatility,
    "sharpe": sharpe
})

factor_table.sort_values(
    "momentum_60d",
    ascending=False
).head(10)

,momentum_20d,momentum_60d,momentum_120d,volatility,sharpe
Ticker,,,,,
GOOGL,-0.008249,0.250668,0.739746,0.288416,0.926980
WMT,-0.006876,0.087021,0.185359,0.215112,0.772012
KO,-0.010754,0.065069,0.015328,0.177590,0.527174
XOM,0.042988,0.062982,0.061441,0.274172,0.380248
AAPL,-0.050072,0.060125,0.290170,0.288458,0.908110
AMZN,-0.015357,0.044907,0.025775,0.329283,0.912963
JPM,0.046577,0.042176,0.128727,0.271490,0.783892
PEP,-0.024805,0.037393,0.081927,0.190322,0.453860
V,0.063983,0.006083,0.011764,0.242595,0.776346
